# Google Search + Crawl Tester

Nhập API key + CX của Google Custom Search rồi chạy để:
1. Gửi truy vấn tìm kiếm
2. Lấy top URL (giữ nguyên thứ tự)
3. Crawl nhanh nội dung từng URL để kiểm tra text



In [ ]:
import os
import asyncio
from typing import Sequence

import aiohttp
import requests

try:
    from bs4 import BeautifulSoup
except ImportError as exc:  # pragma: no cover
    raise ImportError("pip install beautifulsoup4") from exc

os.environ.setdefault("WEB_SEARCH_SSL", "False")
DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

def google_search(query: str, api_key: str, cx: str, num: int = 5) -> list[dict[str, str]]:
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": api_key,
        "cx": cx,
        "q": query,
        "num": num,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    items = data.get("items", [])
    return [
        {
            "title": item.get("title"),
            "snippet": item.get("snippet"),
            "link": item.get("link"),
        }
        for item in items
    ]

async def _fetch(url: str, ssl: bool = False) -> str:
    timeout = aiohttp.ClientTimeout(total=60)
    connector = aiohttp.TCPConnector(ssl=ssl)
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        async with session.get(url, headers=DEFAULT_HEADERS) as response:
            response.raise_for_status()
            return await response.text()

def _extract_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return "\n".join(chunk.strip() for chunk in soup.stripped_strings if chunk.strip())

async def crawl_urls(urls: Sequence[str], ssl: bool = False) -> list[dict[str, str]]:
    results = []
    for url in urls:
        try:
            html = await _fetch(url, ssl=ssl)
            text = _extract_text(html)
            results.append({"url": url, "html_len": len(html), "text": text})
        except Exception as exc:
            results.append({"url": url, "error": str(exc)})
    return results



In [ ]:
GOOGLE_API_KEY = ""  # <-- điền key vào đây
GOOGLE_CX = ""      # <-- điền CX vào đây
QUERY = "Thủ khoa ngành trí tuệ nhân tạo UET 2025"
TOP_N = 5
SSL_FLAG = os.environ.get("WEB_SEARCH_SSL", "False").lower() in ("true", "1")

if not GOOGLE_API_KEY or not GOOGLE_CX:
    raise ValueError("Cần điền GOOGLE_API_KEY và GOOGLE_CX")

search_results = google_search(QUERY, GOOGLE_API_KEY, GOOGLE_CX, num=TOP_N)
print(f"Query='{QUERY}' | results={len(search_results)}")
for idx, item in enumerate(search_results, start=1):
    print(f"[{idx}] {item['title']}")
    print(f"     URL: {item['link']}")
    print(f"     Snippet: {item['snippet']}")


In [ ]:
urls = [item["link"] for item in search_results]
results = asyncio.run(crawl_urls(urls, ssl=SSL_FLAG))

for idx, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {idx}: {item['url']}")
    if "error" in item:
        print(f"❌ Error: {item['error']}")
    else:
        snippet = item["text"][:1000]
        print(f"HTML length: {item['html_len']:,}")
        print(f"Text preview ({len(snippet)} chars):\n{snippet}")

